# **(EDL - Edit, transform, Load)**

In [1]:
#import numy 
import numpy as np


In [2]:
#impart pandas
import pandas as pd

## Objectives

* Upload raw data, analyse and transform the date to remove inconsistencies

## Inputs

* The data is onlne retail data in csv form that has been moved this directory dataset\raw\Online_Retail.csv

## Outputs

* The output will be converted dat that has been amedned to remove the following:-
***********************

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [3]:
import os
current_dir = os.getcwd()
current_dir

'c:\\vscode_projects\\Hack1_Online_Retail\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [4]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'c:\\vscode_projects\\Hack1_Online_Retail'

# Data extraction

Section 1 content

In [6]:
# set file path
file_path = r'dataset\raw\Online_Retail.csv'

# load csv
df_retail = pd.read_csv(file_path)

#list first 20 rows
df_retail.head(20)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


---

# Check for missleading data and missing data

To get a summary of the data using the describe method.

In [7]:
# copy data set and re-format summary to make it easier to read
summary = df_retail.describe()
summary['Quantity'] = summary['Quantity'].round(0).astype(int)      # integers
summary['CustomerID'] = summary['CustomerID'].round(0).astype(int)  # integers
summary['UnitPrice'] = summary['UnitPrice'].round(2)                # 2 decimals

summary

,Quantity,UnitPrice,CustomerID
count,541909,541909.00,541909
mean,10,4.61,15288
std,218,96.76,1485
min,-80995,-11062.06,12346
25%,1,1.25,14367
50%,3,2.08,15287
75%,10,4.13,16255
max,80995,38970.00,18287


Look for missing data using isnull

In [8]:
missing_data = df_retail[df_retail.isnull().any(axis=1)]
missing_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,15287,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,15287,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,15287,United Kingdom
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,15287,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,15287,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,15287,United Kingdom


In [9]:
# data where the description or customer ID is missing can not be used effectively as the products can not be identified and it cannot be grouped to a customer

count_missing = df_retail[["CustomerID","Description"]].isnull().sum()
count_missing


CustomerID        0
Description    1454
dtype: int64

# Data Analysis

•	The describe function has displayed all data items as floats, there are also too many decimal points to reflect the true value of the data. It is rare to get decimals in quantities for retail, as most retail items are sold in single units. This will require the conversion of the data types.
•	The min is a negative number for quantity; this suggests that returns orders are included in the data set or that there are mistakes in the data. 
•	The max for quantity is extremely high for a retail customer order, this suggests that there are outliers. 
•	The min is a negative number for unit price; this suggests that returns orders are included in the data set or that there are mistakes in the data. 
•	The mean for quality is large for retail customer orders. This suggests that there are outliers in the data.
•	The data with missing descriptions will need to be dropped from the data set


The following action will be taken to clean the data 
1 - Convert quantity to integer
2 - Convert unit price to   float
3 - Convert customer Id to a string
4 - Convert InvoiceDate to date to date and time
5 - Filter out negative quantities for quantity and unit price
6 – Remove missing values in customerID and description
7 - Remove outliers in quantity


In [13]:
#Load the data into a new dataframe for cleaning

df_clean_retail = df_retail.copy()

# 1 Ensure Quantity is integer
df_clean_retail["Quantity"] = df_clean_retail["Quantity"].astype(int)

# 2 Ensure UnitPrice is float (already float, but safe to enforce)
df_clean_retail["UnitPrice"] = df_clean_retail["UnitPrice"].astype(float)

# 3 Convert CustomerID to string (since it's an identifier, not numeric)
df_clean_retail["CustomerID"] = df_clean_retail["CustomerID"].astype("string")

# 4 convert invoice date and time to datetime
df_clean_retail["InvoiceDate"] = pd.to_datetime(df_clean_retail["InvoiceDate"], errors="coerce")

# 5 filter out negative quantities for quantity and unit price
df_clean_retail = df_clean_retail[df_clean_retail["Quantity"] > 0]
df_clean_retail = df_clean_retail[df_clean_retail["UnitPrice"] > 0]

# 6 filter out missing values
df_clean_retail = df_clean_retail.dropna()


# Review Cleaned Data set 

In [16]:
df_clean_retail.head(20)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


In [40]:
# List summary and re-format to make it easier to read
# Exclude InvoiceDate & CustomerID from numeric summary
new_summary = df_clean_retail.drop(columns=['InvoiceDate', 'CustomerID']).describe().round(2)

# Add counts for excluded columns
new_summary.loc['count', 'InvoiceDate'] = df_clean_retail['InvoiceDate'].count()
new_summary.loc['count', 'CustomerID']  = df_clean_retail['CustomerID'].count()

new_summary


,Quantity,UnitPrice,InvoiceDate,CustomerID
count,530104.00,530104.00,530104.0,530104.0
mean,10.54,3.91,NaN,NaN
std,155.52,35.92,NaN,NaN
min,1.00,0.00,NaN,NaN
25%,1.00,1.25,NaN,NaN
50%,3.00,2.08,NaN,NaN
75%,10.00,4.13,NaN,NaN
max,80995.00,13541.33,NaN,NaN


In [41]:
#Check for missing data using isnull
missing_data2 = df_clean_retail[df_clean_retail.isnull().any(axis=1)]
missing_data2

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [45]:
df_clean_retail


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,3,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,3,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,1,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4,12680,France


The count has reduced from 541,909 to 530,104.
The cleansing has removed 11,805 rows of rows containing nulls or negative values.
The cleaned data set now has 0 records with missing data.

# Check for duplicates

The unique dat in the retail 

In [44]:
duplicate_data = df_clean_retail[df_clean_retail.duplicated(subset = ['InvoiceNo','CustomerID', 'StockCode'])]
duplicate_data


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
125,536381,71270,PHOTO CLIP LINE,3,2010-12-01 09:41:00,1,15311,United Kingdom
498,536409,90199C,5 STRAND GLASS NECKLACE CRYSTAL,1,2010-12-01 11:45:00,6,17908,United Kingdom
502,536409,85116,BLACK CANDELABRA T-LIGHT HOLDER,5,2010-12-01 11:45:00,2,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1,17908,United Kingdom
525,536409,90199C,5 STRAND GLASS NECKLACE CRYSTAL,2,2010-12-01 11:45:00,6,17908,United Kingdom
...,...,...,...,...,...,...,...,...
541692,581538,22992,REVOLVER WOODEN RULER,1,2011-12-09 11:34:00,2,14446,United Kingdom
541697,581538,21194,PINK HONEYCOMB PAPER FAN,1,2011-12-09 11:34:00,1,14446,United Kingdom
541698,581538,35004B,SET OF 3 BLACK FLYING DUCKS,1,2011-12-09 11:34:00,5,14446,United Kingdom
541699,581538,22694,WICKER STAR,1,2011-12-09 11:34:00,2,14446,United Kingdom


Add categories to group countries inot regions and to differenciate between producs and non products

# Remove Outliers

Identify range of Outliers and use requirements from the customer ot set max ranges for quantity.  The customer would like amazon fees, non product fees and postage pees to be examined separately to the products.



In [55]:
#Stores outliers in data set - These have been identified as where the quantities are  greater than 3200 and unit price is less than 700
outliers = df_clean_retail[
    (df_clean_retail['Quantity'] <= 0) | (df_clean_retail['Quantity'] > 3200) |
    (df_clean_retail['UnitPrice'] <= 0) | (df_clean_retail['UnitPrice'] > 700)
]
# removes outliers from data set
df_no_outliers = df_clean_retail.drop(outliers.index)

outliers


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6165,536876,DOT,DOTCOM POSTAGE,1,2010-12-03 11:36:00,887.52,15287,United Kingdom
10812,537237,DOT,DOTCOM POSTAGE,1,2010-12-06 09:58:00,863.74,15287,United Kingdom
11381,537240,DOT,DOTCOM POSTAGE,1,2010-12-06 10:08:00,940.87,15287,United Kingdom
13924,537434,DOT,DOTCOM POSTAGE,1,2010-12-06 16:57:00,950.99,15287,United Kingdom
14392,537534,M,Manual,1,2010-12-07 11:48:00,924.59,15287,United Kingdom
...,...,...,...,...,...,...,...,...
537254,581238,DOT,DOTCOM POSTAGE,1,2011-12-08 10:53:00,1683.75,15287,United Kingdom
539368,581439,DOT,DOTCOM POSTAGE,1,2011-12-08 16:30:00,938.59,15287,United Kingdom
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom
540908,581492,DOT,DOTCOM POSTAGE,1,2011-12-09 10:03:00,933.17,15287,United Kingdom


Summary of data

In [57]:
# List summary and re-format to make it easier to read, separate out non numeric and create counts
clean_summary = df_no_outliers.drop(columns=['InvoiceDate', 'CustomerID']).describe().round(2)

# Add counts for excluded columns
clean_summary.loc['count', 'InvoiceDate'] = df_no_outliers['InvoiceDate'].count()
clean_summary.loc['count', 'CustomerID']  = df_no_outliers['CustomerID'].count()

clean_summary


,Quantity,UnitPrice,InvoiceDate,CustomerID
count,530002.00,530002.00,530002.0,530002.0
mean,10.23,3.60,NaN,NaN
std,36.36,10.28,NaN,NaN
min,1.00,0.00,NaN,NaN
25%,1.00,1.25,NaN,NaN
50%,3.00,2.08,NaN,NaN
75%,10.00,4.13,NaN,NaN
max,3186.00,700.00,NaN,NaN


* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.